# Modelado Random Forest (2018–2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.

El objetivo es desarrollar, entrenar y evaluar un modelo **Random Forest** para la predicción de severidad en hechos de tránsito a partir del dataset procesado durante las fases de EDA y ETL. Esta implementación constituye la primera iteración dentro del conjunto de modelos candidatos del proyecto.


In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, accuracy_score,
                             roc_curve, precision_recall_curve)
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de datos

In [2]:
# -- Carga ----------------------------------------------
train = pd.read_parquet('../data/clean/train.parquet')
test  = pd.read_parquet('../data/clean/test.parquet')

FEATURES = ['tipo_eve','tipo_veh','g_hora_5','dia_sem_ocu',
            'sexo_per','edad_quinquenales','mayor_menor','depto_ocu']
TARGET = 'fall_les'

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f'Train : {X_train.shape}')
print(f'Test  : {X_test.shape}')
print(f'\nDistribución target (test):')
print(y_test.value_counts().rename({1:"Fallecido", 2:"Lesionado"}))

Train : (57553, 8)
Test  : (14389, 8)

Distribución target (test):
fall_les
Lesionado    11641
Fallecido     2748
Name: count, dtype: int64


## 2. Entrenamiento Random Forest

In [3]:
# -- Entrenamiento ----------------------------------------------
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print('✓ Modelo entrenado correctamente')

✓ Modelo entrenado correctamente


## 3. Evaluación del modelo

In [4]:
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 0]   # probabilidad de Fallecido (clase 1)

# Convertir y_test a binario: 1 = Fallecido, 0 = Lesionado
y_test_bin = (y_test == 1).astype(int)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='weighted')
auc = roc_auc_score(y_test_bin, y_proba)
cm  = confusion_matrix(y_test, y_pred)

print('=== Random Forest ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'ROC-AUC  : {auc:.4f}')

=== Random Forest ===
Accuracy : 0.6949
F1-Score : 0.7233
ROC-AUC  : 0.7336


In [5]:
# -- Visualización 1: Métricas generales ----------------------------------------------
fig_metricas = go.Figure(go.Bar(
    x=['Accuracy', 'F1-Score', 'ROC-AUC'],
    y=[acc, f1, auc],
    text=[f'{acc:.4f}', f'{f1:.4f}', f'{auc:.4f}'],
    textposition='outside',
    marker_color=['#5B8DEF', '#F4A261', '#2EC4B6'],
    width=0.4
))
fig_metricas.update_layout(
    title='Métricas de evaluación : Random Forest',
    yaxis=dict(range=[0, 1], title='Valor'),
    xaxis_title='Métrica',
    height=400,
    template='plotly_white'
)
fig_metricas.show()

In [6]:
# -- Visualización 2: Matriz de confusión ----------------------------------------------
cm_labels = ['Fallecido', 'Lesionado']
fig_cm = px.imshow(
    cm,
    labels=dict(x='Predicción', y='Real', color='Cantidad'),
    x=cm_labels,
    y=cm_labels,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Matriz de confusión : Random Forest'
)
fig_cm.update_layout(height=400, template='plotly_white')
fig_cm.show()

In [7]:
# -- Visualización 3: Curva ROC ----------------------------------------------
fpr, tpr, _ = roc_curve(y_test_bin, y_proba)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'Random Forest (AUC = {auc:.4f})',
    line=dict(color='#5B8DEF', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1],
    mode='lines',
    name='Baseline (AUC = 0.5)',
    line=dict(color='gray', width=1.5, dash='dash')
))
fig_roc.update_layout(
    title='Curva ROC : Random Forest',
    xaxis_title='Tasa de Falsos Positivos',
    yaxis_title='Tasa de Verdaderos Positivos',
    height=450,
    template='plotly_white',
    legend=dict(x=0.6, y=0.1)
)
fig_roc.show()

In [8]:
# -- Visualización 4: Importancia de features ----------------------------------------------
importancias = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

LABELS = {
    'tipo_eve'         : 'Tipo de evento',
    'tipo_veh'         : 'Tipo de vehículo',
    'g_hora_5'         : 'Grupo horario',
    'dia_sem_ocu'      : 'Día de la semana',
    'sexo_per'         : 'Sexo',
    'edad_quinquenales': 'Grupo de edad',
    'mayor_menor'      : 'Mayor / Menor edad',
    'depto_ocu'        : 'Departamento'
}

fig_imp = go.Figure(go.Bar(
    x=importancias.values,
    y=[LABELS[f] for f in importancias.index],
    orientation='h',
    marker_color='#5B8DEF',
    text=[f'{v:.4f}' for v in importancias.values],
    textposition='auto',
    textfont=dict(size=11)
))
fig_imp.update_layout(
    title='Importancia de features : Random Forest',
    xaxis=dict(title='Importancia (Gini)', range=[0, max(importancias.values) * 1.25]),
    yaxis_title='Feature',
    height=450,
    template='plotly_white'
)
fig_imp.show()

### Hallazgo — Importancia de features

| Feature | Importancia |
|---|---|
| Departamento | 0.2342 : más predictora |
| Tipo de evento | 0.1849 |
| Grupo de edad | 0.1655 |
| Tipo de vehículo | 0.1276 |
| Sexo | 0.1104 |
| Día de la semana | 0.0966 |
| Grupo horario | 0.0544 |
| Mayor / Menor edad | 0.0265 : menos predictora |

Departamento fue la variable con mayor importancia según el índice de Gini (23.4%), seguida por tipo de evento (18.5%). Este resultado sugiere que la ubicación geográfica constituye uno de los principales factores asociados con la severidad del siniestro en el conjunto de datos analizado.


In [9]:
# -- Visualización 5: Distribución de probabilidades predichas ----------------------------------------------
df_proba = pd.DataFrame({
    'probabilidad': y_proba,
    'real': y_test.map({1:'Fallecido', 2:'Lesionado'})
})

fig_dist = px.histogram(
    df_proba,
    x='probabilidad',
    color='real',
    nbins=50,
    barmode='overlay',
    opacity=0.7,
    color_discrete_map={'Fallecido':'#E63946', 'Lesionado':'#5B8DEF'},
    title='Distribución de probabilidades predichas : Random Forest',
    labels={'probabilidad':'Probabilidad predicha (clase Fallecido)',
            'real':'Clase real'}
)
fig_dist.update_layout(height=420, template='plotly_white')
fig_dist.show()

### Análisis de distribución de probabilidades

Las distribuciones de Fallecido y Lesionado siguen solapándose
en la zona 0.3–0.7, lo cual es esperado dado que las features
son categóricas y el desbalance de clases es 4.2:1.
, la distribución es más honesta con los datos reales —
el modelo asigna probabilidades más conservadoras y distribuidas.

In [10]:
# -- Guardar resultados para comparación final ----------------------------------------------
from sklearn.metrics import precision_score, recall_score

precision_fallecido = precision_score(y_test_bin, (y_pred == 1).astype(int))
recall_fallecido = recall_score(y_test_bin, (y_pred == 1).astype(int))

resultados_rf = {
    'modelo'   : 'Random Forest',
    'accuracy' : round(acc, 4),
    'f1_score' : round(f1, 4),
    'roc_auc'  : round(auc, 4),
    'precision_fallecido': round(precision_fallecido, 4),
    'recall_fallecido'   : round(recall_fallecido, 4),
}

import json, os
os.makedirs('../data/models', exist_ok=True)
with open('../data/models/resultados_rf.json', 'w') as f:
    json.dump(resultados_rf, f, indent=2)

print('✓ Resultados guardados en data/models/resultados_rf.json')
print(f'\nResumen Random Forest:')
for k, v in resultados_rf.items():
    print(f'  {k:<25} {v}')

✓ Resultados guardados en data/models/resultados_rf.json

Resumen Random Forest:
  modelo                    Random Forest
  accuracy                  0.6949
  f1_score                  0.7233
  roc_auc                   0.7336
  precision_fallecido       0.3382
  recall_fallecido          0.6245


In [11]:
# -- Guardar modelo entrenado ----------------------------------------------
import joblib
import os

os.makedirs('../data/models', exist_ok=True)
joblib.dump(rf, '../data/models/random_forest.pkl')
print('✓ Modelo guardado en data/models/random_forest.pkl')

✓ Modelo guardado en data/models/random_forest.pkl


## 4. Resumen del modelo

| Métrica | Valor |
|---|---|
| Accuracy | 69.49% |
| F1-Score (weighted) | 72.33% |
| ROC-AUC | 73.36% |
| Precision Fallecido | 33.82% |
| Recall Fallecido | 62.45% |


## 5. Split de validación para selección de hiperparámetros

`X_train_final` y `X_val` ya no se recalculan aquí: se cargan desde `train_final.parquet` y `val.parquet`, generados una sola vez en `05_dataset_modeling.ipynb` (mismo `train_test_split(test_size=0.25, random_state=42, stratify=y_train)`). `X_test` no se toca en ningún momento de esta sección — la búsqueda de hiperparámetros se evalúa exclusivamente contra `X_val`.

In [12]:
# -- Cargar split de validación centralizado (generado en 05_dataset_modeling.ipynb) ----------------------------------------------
from sklearn.model_selection import ParameterSampler

train_final = pd.read_parquet('../data/clean/train_final.parquet')
val = pd.read_parquet('../data/clean/val.parquet')

X_train_final = train_final[FEATURES]
y_train_final = train_final[TARGET]
X_val = val[FEATURES]
y_val = val[TARGET]

total_modelado = len(X_train) + len(X_test)
print(f'Total dataset de modelado (train+test) : {total_modelado:,}')
print(f'X_train_final : {X_train_final.shape[0]:,}  ({X_train_final.shape[0]/total_modelado*100:.2f}% del total)')
print(f'X_val         : {X_val.shape[0]:,}  ({X_val.shape[0]/total_modelado*100:.2f}% del total)')
print(f'X_test        : {X_test.shape[0]:,}  ({X_test.shape[0]/total_modelado*100:.2f}% del total)')

Total dataset de modelado (train+test) : 71,942
X_train_final : 43,164  (60.00% del total)
X_val         : 14,389  (20.00% del total)
X_test        : 14,389  (20.00% del total)


## 6. Búsqueda de hiperparámetros (25 combinaciones aleatorias, evaluadas en X_val)

In [13]:
# -- Búsqueda aleatoria manual: entrena SOLO con X_train_final, evalúa F1 weighted SOLO con X_val ----------------------------------------------
param_grid_rf = {
    'n_estimators'    : [100, 150, 200, 300, 400],
    'max_depth'       : [10, 15, 20, 25, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features'    : ['sqrt', 'log2', 0.3, 0.5],
}
sampler_rf = list(ParameterSampler(param_grid_rf, n_iter=25, random_state=42))

resultados_busqueda_rf = []
for params in sampler_rf:
    modelo_tmp = RandomForestClassifier(**params, class_weight='balanced', random_state=42, n_jobs=-1)
    modelo_tmp.fit(X_train_final, y_train_final)
    pred_val = modelo_tmp.predict(X_val)
    f1_val = f1_score(y_val, pred_val, average='weighted')
    resultados_busqueda_rf.append({**params, 'f1_val': f1_val})

df_busqueda_rf = pd.DataFrame(resultados_busqueda_rf).sort_values('f1_val', ascending=False).reset_index(drop=True)
print(df_busqueda_rf.to_string())

mejor_rf = max(resultados_busqueda_rf, key=lambda d: d['f1_val'])
mejores_params_rf = {k: v for k, v in mejor_rf.items() if k != 'f1_val'}
print(f'\nMejor F1 (validación): {mejor_rf["f1_val"]:.4f}')
print(f'Mejores hiperparámetros (Random Forest): {mejores_params_rf}')

    n_estimators  min_samples_leaf max_features  max_depth    f1_val
0            150                 1          0.3        NaN  0.749278
1            100                 1          0.3       25.0  0.746237
2            400                 1         sqrt       15.0  0.744621
3            100                 5          0.5        NaN  0.744021
4            150                 5         log2       25.0  0.742663
5            400                 5         sqrt        NaN  0.740666
6            400                 5          0.3       20.0  0.738277
7            150                10          0.5        NaN  0.731476
8            150                10          0.5       20.0  0.729652
9            150                 5          0.3       15.0  0.728822
10           150                10         log2       25.0  0.726844
11           100                10          0.3       20.0  0.723369
12           400                10         sqrt       15.0  0.717733
13           200                10

## 7. Reentrenamiento final (train_final + val) y evaluación en test

In [14]:
# -- Reentrenar UNA sola vez con los mejores hiperparámetros sobre X_train completo, evaluar UNA sola vez en X_test ----------------------------------------------
from sklearn.metrics import precision_score, recall_score
import joblib, os

rf_v2 = RandomForestClassifier(**mejores_params_rf, class_weight='balanced', random_state=42, n_jobs=-1)
rf_v2.fit(X_train, y_train)

pred_train_v2 = rf_v2.predict(X_train)
pred_test_v2  = rf_v2.predict(X_test)
proba_test_v2 = rf_v2.predict_proba(X_test)[:, list(rf_v2.classes_).index(1)]  # 1 = Fallecido (Deceased)
y_test_bin_v2 = (y_test == 1).astype(int)

acc_train_v2 = accuracy_score(y_train, pred_train_v2)
acc_test_v2  = accuracy_score(y_test, pred_test_v2)
f1_test_v2   = f1_score(y_test, pred_test_v2, average='weighted')
auc_test_v2  = roc_auc_score(y_test_bin_v2, proba_test_v2)
prec_fall_v2 = precision_score(y_test_bin_v2, (pred_test_v2 == 1).astype(int))
rec_fall_v2  = recall_score(y_test_bin_v2, (pred_test_v2 == 1).astype(int))
overfit_v2   = acc_train_v2 - acc_test_v2

print('=== Random Forest v2 (hiperparámetros seleccionados con validación) ===')
print(f'Hiperparámetros      : {mejores_params_rf}')
print(f'Accuracy train       : {acc_train_v2:.4f}')
print(f'Accuracy test        : {acc_test_v2:.4f}')
print(f'F1 weighted (test)   : {f1_test_v2:.4f}')
print(f'ROC-AUC (test)       : {auc_test_v2:.4f}')
print(f'Precision Deceased   : {prec_fall_v2:.4f}')
print(f'Recall Deceased      : {rec_fall_v2:.4f}')
print(f'Overfitting (train-test acc) : {overfit_v2:.4f}')

os.makedirs('../data/models', exist_ok=True)
joblib.dump(rf_v2, '../data/models/random_forest_v2.pkl')
print('\n✓ Modelo guardado en data/models/random_forest_v2.pkl (no sobrescribe random_forest.pkl)')

=== Random Forest v2 (hiperparámetros seleccionados con validación) ===
Hiperparámetros      : {'n_estimators': 150, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': None}
Accuracy train       : 0.8866
Accuracy test        : 0.7502
F1 weighted (test)   : 0.7492
ROC-AUC (test)       : 0.6615
Precision Deceased   : 0.3426
Recall Deceased      : 0.3352
Overfitting (train-test acc) : 0.1363



✓ Modelo guardado en data/models/random_forest_v2.pkl (no sobrescribe random_forest.pkl)


## 8. Búsqueda de hiperparámetros con f1_macro (v3)

Misma rejilla y mismas 25 combinaciones (`random_state=42`) que en la sección 6, pero seleccionando por `f1_macro` en vez de `f1_weighted`, y registrando también el Recall de "Deceased" (Fallecido) por candidato en `X_val`.

In [15]:
# -- Búsqueda con f1_macro (mismo espacio de 25 combinaciones); registra también Recall de Deceased por candidato ----------------------------------------------
sampler_rf_macro = list(ParameterSampler(param_grid_rf, n_iter=25, random_state=42))

resultados_busqueda_rf_macro = []
for params in sampler_rf_macro:
    modelo_tmp = RandomForestClassifier(**params, class_weight='balanced', random_state=42, n_jobs=-1)
    modelo_tmp.fit(X_train_final, y_train_final)
    pred_val = modelo_tmp.predict(X_val)
    f1_macro_val = f1_score(y_val, pred_val, average='macro')
    rec_deceased_val = recall_score((y_val == 1).astype(int), (pred_val == 1).astype(int))
    resultados_busqueda_rf_macro.append({**params, 'f1_macro_val': f1_macro_val, 'recall_deceased_val': rec_deceased_val})

df_busqueda_rf_macro = pd.DataFrame(resultados_busqueda_rf_macro).sort_values('f1_macro_val', ascending=False).reset_index(drop=True)
print(df_busqueda_rf_macro.to_string())

mejor_rf_macro = max(resultados_busqueda_rf_macro, key=lambda d: d['f1_macro_val'])
mejores_params_rf_v3 = {k: v for k, v in mejor_rf_macro.items() if k not in ('f1_macro_val', 'recall_deceased_val')}
print(f'\nMejor F1-macro (validación): {mejor_rf_macro["f1_macro_val"]:.4f}  |  Recall Deceased (val): {mejor_rf_macro["recall_deceased_val"]:.4f}')
print(f'Mejores hiperparámetros v3 (Random Forest): {mejores_params_rf_v3}')

    n_estimators  min_samples_leaf max_features  max_depth  f1_macro_val  recall_deceased_val
0            150                 5         log2       25.0      0.619788             0.525837
1            400                 5         sqrt        NaN      0.619230             0.535662
2            100                 5          0.5        NaN      0.618869             0.510917
3            400                 1         sqrt       15.0      0.618223             0.502911
4            400                 5          0.3       20.0      0.617820             0.542576
5            150                10          0.5        NaN      0.616358             0.579330
6            150                 5          0.3       15.0      0.614107             0.582242
7            150                10          0.5       20.0      0.613823             0.574600
8            150                10         log2       25.0      0.611909             0.580786
9            100                10          0.3       20.0  

## 9. Reentrenamiento final v3 (train_final + val) y evaluación en test

In [16]:
# -- Reentrenar v3 UNA sola vez con los hiperparámetros seleccionados por f1_macro, evaluar UNA sola vez en X_test ----------------------------------------------
rf_v3 = RandomForestClassifier(**mejores_params_rf_v3, class_weight='balanced', random_state=42, n_jobs=-1)
rf_v3.fit(X_train, y_train)

pred_train_v3 = rf_v3.predict(X_train)
pred_test_v3  = rf_v3.predict(X_test)
proba_test_v3 = rf_v3.predict_proba(X_test)[:, list(rf_v3.classes_).index(1)]
y_test_bin_v3 = (y_test == 1).astype(int)

acc_train_v3 = accuracy_score(y_train, pred_train_v3)
acc_test_v3  = accuracy_score(y_test, pred_test_v3)
f1w_test_v3  = f1_score(y_test, pred_test_v3, average='weighted')
f1m_test_v3  = f1_score(y_test, pred_test_v3, average='macro')
auc_test_v3  = roc_auc_score(y_test_bin_v3, proba_test_v3)
prec_fall_v3 = precision_score(y_test_bin_v3, (pred_test_v3 == 1).astype(int))
rec_fall_v3  = recall_score(y_test_bin_v3, (pred_test_v3 == 1).astype(int))
overfit_v3   = acc_train_v3 - acc_test_v3

print('=== Random Forest v3 (hiperparámetros seleccionados con f1_macro) ===')
print(f'Hiperparámetros      : {mejores_params_rf_v3}')
print(f'Accuracy train       : {acc_train_v3:.4f}')
print(f'Accuracy test        : {acc_test_v3:.4f}')
print(f'F1 weighted (test)   : {f1w_test_v3:.4f}')
print(f'F1 macro (test)      : {f1m_test_v3:.4f}')
print(f'ROC-AUC (test)       : {auc_test_v3:.4f}')
print(f'Precision Deceased   : {prec_fall_v3:.4f}')
print(f'Recall Deceased      : {rec_fall_v3:.4f}')
print(f'Overfitting (train-test acc) : {overfit_v3:.4f}')

joblib.dump(rf_v3, '../data/models/random_forest_v3.pkl')
print('\n✓ Modelo guardado en data/models/random_forest_v3.pkl')

=== Random Forest v3 (hiperparámetros seleccionados con f1_macro) ===
Hiperparámetros      : {'n_estimators': 150, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 25}
Accuracy train       : 0.7954
Accuracy test        : 0.7290
F1 weighted (test)   : 0.7473
F1 macro (test)      : 0.6261
ROC-AUC (test)       : 0.7235
Precision Deceased   : 0.3593
Recall Deceased      : 0.5349
Overfitting (train-test acc) : 0.0663

✓ Modelo guardado en data/models/random_forest_v3.pkl
